In [3]:
# ============================================
# 电商用户销售与行为分析（Pandas 版）
# ============================================

import pandas as pd

orders = pd.read_csv('data/orders.csv')
products = pd.read_csv('data/products.csv')
users = pd.read_csv('data/users.csv')

In [3]:
# --------------------------------------------
# 第 1 题：商品销量排名
# --------------------------------------------
result1 = (
    pd.merge(orders, products, on='product_id')
    .query('order_status == "已完成"')
    .groupby('product_name')['quantity']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
result1

,product_name,quantity
0,数据线D,19
1,鼠标J,10
2,耳机B,9
3,智能手环M,7
4,手机C,6
5,手机A,6
6,充电宝C,6
7,手机B,5
8,键盘I,5
9,智能手表L,4


In [4]:
# --------------------------------------------
# 第 2 题：商品销售额排名
# --------------------------------------------
merged = pd.merge(orders, products, on='product_id')
merged = merged.query('order_status == "已完成"')
merged['sales'] = merged['quantity'] * merged['price']

result2 = (
    merged.groupby('product_name')['sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
result2

,product_name,sales
0,笔记本H,35996.0
1,手机A,23994.0
2,笔记本G,17997.0
3,手机B,14495.0
4,手机C,9594.0
5,平板F,6598.0
6,智能手表L,6396.0
7,耳机B,2691.0
8,显示器K,2598.0
9,平板E,2299.0


In [5]:
# --------------------------------------------
# 第 3 题：月度销售额
# --------------------------------------------
merged = pd.merge(orders, products, on='product_id')
merged = merged.query('order_status == "已完成"')
merged['sales'] = merged['quantity'] * merged['price']
merged['order_date'] = pd.to_datetime(merged['order_date'])
merged['month'] = merged['order_date'].dt.strftime('%Y-%m')

result3 = (
    merged.groupby('month')['sales']
    .sum()
    .reset_index()
    .sort_values(by='month')
)
result3

,month,sales
0,2024-07,47825.0
1,2024-08,43919.0
2,2024-09,36057.0


In [6]:
# --------------------------------------------
# 第 4 题：用户消费排名
# --------------------------------------------
merged = pd.merge(orders, products, on='product_id')
merged = pd.merge(merged, users, on='user_id')
merged = merged.query('order_status == "已完成"')
merged['sales'] = merged['quantity'] * merged['price']

result4 = (
    merged.groupby(['user_id', 'gender', 'age'])['sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
result4

,user_id,gender,age,sales
0,5,男,35,18375.0
1,13,男,29,14597.0
2,10,女,33,10897.0
3,19,男,38,9998.0
4,11,男,41,9444.0
5,1,男,24,8791.0
6,2,女,31,8096.0
7,3,男,28,7598.0
8,4,女,22,6311.0
9,15,男,25,6198.0


In [10]:
# --------------------------------------------
# 第 5 题：复购率（口径 A - 用户维度）
# --------------------------------------------

# 每个用户下了几单
user_orders = (
    orders.query('order_status == "已完成"')
    .groupby('user_id')
    .size()
    .reset_index(name='sum_orders')
)

# 复购用户数（订单数 >= 2）
repeat_users = len(user_orders[user_orders['sum_orders'] >= 2])

# 总消费用户数
total_users = len(user_orders)

# 复购率
repeat_rate = round(repeat_users / total_users * 100, 2)

print("复购用户数:", repeat_users)
print("总消费用户数:", total_users)
print("复购率:", repeat_rate, "%")

复购用户数: 18
总消费用户数: 19
复购率: 94.74 %


In [13]:
# --------------------------------------------
# 第 6 题：复购率（口径 B - 订单维度）
# --------------------------------------------

# 筛选已完成订单
done = orders.query('order_status == "已完成"')

# 总订单数
total_orders = len(done)

# 下单用户数（去重后有几个用户）
total_users = done['user_id'].nunique()

# 复购订单数 = 总订单 - 首单数
repeat_orders = total_orders - total_users

# 复购率
repeat_rate = round(repeat_orders / total_orders * 100, 2)

print("总订单数:", total_orders)
print("下单用户数:", total_users)
print("复购订单数:", repeat_orders)
print("复购率:", repeat_rate, "%")

总订单数: 54
下单用户数: 19
复购订单数: 35
复购率: 64.81 %


In [17]:
# --------------------------------------------
# 第 7 题：用户分层（ 4 档）
# --------------------------------------------

merged = pd.merge(orders, products, on='product_id')
trip_merged = pd.merge(merged, users, on='user_id')
trip_merged = trip_merged.query('order_status == "已完成"')
trip_merged['sales'] = trip_merged['price'] * trip_merged['quantity']

user_spent = (
    trip_merged.groupby(['user_id', 'gender', 'age'])['sales']
    .sum()
    .reset_index()
)

user_spent['tier'] = pd.qcut(
    user_spent['sales'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

user_spent = user_spent.sort_values(by='sales', ascending=False)
user_spent

,user_id,gender,age,sales,tier
4,5,男,35,18375.0,Q4
12,13,男,29,14597.0,Q4
9,10,女,33,10897.0,Q4
17,19,男,38,9998.0,Q4
10,11,男,41,9444.0,Q4
0,1,男,24,8791.0,Q3
1,2,女,31,8096.0,Q3
2,3,男,28,7598.0,Q3
3,4,女,22,6311.0,Q3
13,15,男,25,6198.0,Q2


In [19]:
# --------------------------------------------
# 第 8 题：品类销售分析
# --------------------------------------------

merged = pd.merge(orders, products, on='product_id').query('order_status == "已完成"')
merged['sales'] = merged['price'] * merged['quantity']

result = (
    merged.groupby('category').agg(
        总销量=('quantity', 'sum'),
        总销售额=('sales', 'sum'),
        订单数=('order_id', 'count')
    )
    .reset_index()
    .sort_values(by='总销售额', ascending=False)
)
result

,category,总销量,总销售额,订单数
4,电脑,7,53993.0,7
1,手机,17,48083.0,16
0,平板,3,8897.0,3
3,智能穿戴,11,8139.0,7
5,电脑外设,17,4483.0,8
2,数码配件,34,4206.0,13


In [4]:
# --------------------------------------------
# 第 9 题：年龄分层消费
# --------------------------------------------
merged = pd.merge(orders, products, on='product_id')
trip_merged = pd.merge(merged, users, on='user_id')
trip_merged = trip_merged.query('order_status == "已完成"')
trip_merged['sales'] = trip_merged['price'] * trip_merged['quantity']

# 先按用户聚合
user_spend = (
    trip_merged.groupby(['user_id', 'age'])['sales']
    .sum()
    .reset_index()
)

# 按年龄段分箱
user_spend['age_tier'] = pd.cut(
    user_spend['age'],
    bins=[0, 25, 35, 45, 100],
    labels=['18-24', '25-34', '35-44', '45+']
)

# 按年龄段聚合
result = (
    user_spend.groupby('age_tier', observed=True).agg(
        用户数=('user_id', 'count'),
        总消费=('sales', 'sum'),
        人均消费=('sales', 'mean')
    )
    .reset_index()
)
result

,age_tier,用户数,总消费,人均消费
0,18-24,6,27833.0,4638.833333
1,25-34,11,80526.0,7320.545455
2,35-44,2,19442.0,9721.000000


In [2]:
# --------------------------------------------
# 第 10 题：月度复购率趋势
# --------------------------------------------

done = orders.query('order_status == "已完成"').copy()
done['order_date'] = pd.to_datetime(done['order_date'])
done['month'] = done['order_date'].dt.strftime('%Y-%m')

# 月 + 用户 级：每个用户每月下了几单
month_order = (
    done.groupby(['month', 'user_id'])
    .size()
    .reset_index(name='order_count')
)

# 按月聚合
result = (
    month_order.groupby('month')
    .agg(
        总用户数=('user_id', 'count'),
        复购用户数=('order_count', lambda x: (x >= 2).sum())
    )
    .reset_index()
)

# 复购率
result['复购率'] = round(result['复购用户数'] / result['总用户数'] * 100, 2)
result

,month,总用户数,复购用户数,复购率
0,2024-07,12,7,58.33
1,2024-08,17,1,5.88
2,2024-09,14,0,0.00


In [6]:
# --------------------------------------------
# 第 11 题：用户消费间隔分析
# --------------------------------------------

done = orders.query('order_status == "已完成"').copy()
done['order_date'] = pd.to_datetime(done['order_date'])
done = done.sort_values(by=['user_id', 'order_date'])

done['prev_date'] = done.groupby('user_id')['order_date'].shift(1)
done['gap_days'] = (done['order_date'] - done['prev_date']).dt.days

result = (
    done.dropna(subset=['gap_days'])
    .groupby('user_id')
    .agg(
        repeat_count=('gap_days', 'count'),
        avg_gap_days=('gap_days', 'mean')
    )
    .reset_index()
)
result

,user_id,repeat_count,avg_gap_days
0,1,4,15.500000
1,2,3,20.000000
2,3,1,14.000000
3,4,4,14.250000
4,5,3,19.000000
5,6,1,15.000000
6,7,3,18.333333
7,8,2,12.000000
8,9,2,11.000000
9,10,2,24.500000


In [7]:
# 整体平均间隔
overall = done['gap_days'].dropna().mean()
print(f"整体平均间隔：{round(overall, 1)}天")

整体平均间隔：18.1天
